# Reinforcement Learning for Global Equity ETF Allocation

This notebook implements a PPO-based RL agent that dynamically allocates across global equity ETFs using macro signals.

**Key Features:**
- State: Momentum features from TLT, GLD, USO, SLV, VIXY, IBIT
- Actions: Allocate to SPY, EWJ, INDA, MCHI, EZU, or BIL (cash)
- Reward: Risk-adjusted returns (Sharpe-like)
- Algorithm: Proximal Policy Optimization (PPO)

In [91]:
import pandas as pd
import datetime as dt
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable, Dict
from trading_engine.core import (
    read_data, create_model_state, orchestrate_model_backtests,
    orchestrate_model_simulations, orchestrate_portfolio_backtests,
    orchestrate_portfolio_simulations
)
from trading_engine.models import MODELS
from trading_engine.optimizers import OPTIMIZERS
from common.constants import ProcessingMode

# RL imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

## 1. PPO Policy Network

In [92]:
class PolicyNetwork(nn.Module):
    """Neural network that maps state features to action probabilities."""
    
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 128):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=-1)
        
    def forward(self, state):
        x = self.relu(self.fc1(state))
        x = self.relu(self.fc2(x))
        action_probs = self.softmax(self.fc3(x))
        return action_probs


class ValueNetwork(nn.Module):
    """Neural network that estimates state values for advantage calculation."""
    
    def __init__(self, state_dim: int, hidden_dim: int = 128):
        super(ValueNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)
        self.relu = nn.ReLU()
        
    def forward(self, state):
        x = self.relu(self.fc1(state))
        x = self.relu(self.fc2(x))
        value = self.fc3(x)
        return value

## 2. PPO Agent

In [ ]:
class PPOAgent:
    """Proximal Policy Optimization agent for ETF allocation with improved stability."""
    
    def __init__(self, state_dim: int, action_dim: int, 
                 lr_policy: float = 3e-4, lr_value: float = 1e-3,
                 gamma: float = 0.95, epsilon_clip: float = 0.2,
                 gae_lambda: float = 0.95, hidden_dim: int = 128):
        self.gamma = gamma
        self.epsilon_clip = epsilon_clip
        self.gae_lambda = gae_lambda
        self.action_dim = action_dim
        
        # Networks
        self.policy = PolicyNetwork(state_dim, action_dim, hidden_dim)
        self.value = ValueNetwork(state_dim, hidden_dim)
        
        # Optimizers
        self.policy_optimizer = optim.Adam(self.policy.parameters(), lr=lr_policy)
        self.value_optimizer = optim.Adam(self.value.parameters(), lr=lr_value)
        
        # Storage for trajectories
        self.states = []
        self.actions = []
        self.rewards = []
        self.action_probs = []
        self.is_terminals = []
        
    def select_action(self, state, training=True):
        """Select action based on current policy."""
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        
        with torch.no_grad():
            action_probs = self.policy(state_tensor)
        
        if training:
            # Sample from distribution during training (exploration)
            dist = Categorical(action_probs)
            action = dist.sample()
        else:
            # Choose best action during inference (exploitation)
            action = torch.argmax(action_probs, dim=1)
        
        return action.item(), action_probs[0, action.item()].item()
    
    def store_transition(self, state, action, reward, action_prob, is_terminal):
        """Store experience tuple."""
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.action_probs.append(action_prob)
        self.is_terminals.append(is_terminal)
    
    def compute_advantages(self):
        """Compute GAE (Generalized Advantage Estimation) with improved stability."""
        if len(self.states) == 0:
            return np.array([]), np.array([])
        
        states_tensor = torch.FloatTensor(np.array(self.states))
        
        with torch.no_grad():
            values = self.value(states_tensor).numpy().flatten()
        
        advantages = []
        returns = []
        
        # Compute discounted returns and advantages using GAE
        gae = 0
        next_value = 0
        
        for t in reversed(range(len(self.rewards))):
            if self.is_terminals[t]:
                next_value = 0
                gae = 0
            
            # TD error: δ = r + γV(s') - V(s)
            delta = self.rewards[t] + self.gamma * next_value - values[t]
            # GAE: A = δ + γλδ' + ...
            gae = delta + self.gamma * self.gae_lambda * gae
            
            advantages.insert(0, gae)
            returns.insert(0, gae + values[t])
            next_value = values[t]
        
        # Normalize advantages for stability
        advantages = np.array(advantages)
        if len(advantages) > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        return advantages, np.array(returns)
    
    def update(self, epochs=4, batch_size=64):
        """Update policy and value networks using PPO with stability improvements."""
        if len(self.states) < batch_size:
            return
        
        # Compute advantages
        advantages, returns = self.compute_advantages()
        
        if len(advantages) == 0:
            self.clear_memory()
            return
        
        # Convert to tensors
        states_tensor = torch.FloatTensor(np.array(self.states))
        actions_tensor = torch.LongTensor(np.array(self.actions))
        old_probs_tensor = torch.FloatTensor(np.array(self.action_probs))
        advantages_tensor = torch.FloatTensor(advantages)
        returns_tensor = torch.FloatTensor(returns)
        
        # PPO update for multiple epochs
        for _ in range(epochs):
            # Mini-batch update
            indices = np.arange(len(self.states))
            np.random.shuffle(indices)
            
            for start in range(0, len(self.states), batch_size):
                end = min(start + batch_size, len(self.states))
                batch_indices = indices[start:end]
                
                if len(batch_indices) < 2:  # Skip tiny batches
                    continue
                
                # Get batch
                batch_states = states_tensor[batch_indices]
                batch_actions = actions_tensor[batch_indices]
                batch_old_probs = old_probs_tensor[batch_indices]
                batch_advantages = advantages_tensor[batch_indices]
                batch_returns = returns_tensor[batch_indices]
                
                # Evaluate current policy
                action_probs = self.policy(batch_states)
                dist = Categorical(action_probs)
                
                # Compute log probs for stability
                new_log_probs = dist.log_prob(batch_actions)
                old_log_probs = torch.log(batch_old_probs + 1e-8)
                
                # Ratio for PPO (using log space for numerical stability)
                ratio = torch.exp(new_log_probs - old_log_probs)
                
                # PPO clipped objective
                surr1 = ratio * batch_advantages
                surr2 = torch.clamp(ratio, 1 - self.epsilon_clip, 1 + self.epsilon_clip) * batch_advantages
                
                # Add entropy bonus for exploration
                entropy = dist.entropy().mean()
                policy_loss = -torch.min(surr1, surr2).mean() - 0.01 * entropy
                
                # Value loss with clipping
                value_pred = self.value(batch_states).squeeze()
                value_loss = nn.MSELoss()(value_pred, batch_returns)
                
                # Update policy
                self.policy_optimizer.zero_grad()
                policy_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
                self.policy_optimizer.step()
                
                # Update value
                self.value_optimizer.zero_grad()
                value_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.value.parameters(), 0.5)
                self.value_optimizer.step()
        
        # Clear trajectories
        self.clear_memory()
    
    def clear_memory(self):
        """Clear stored trajectories."""
        self.states = []
        self.actions = []
        self.rewards = []
        self.action_probs = []
        self.is_terminals = []

## 3. RL Model Class for Trading Engine Integration

This class follows the same pattern as other models in trading_model0, making it easy to use with orchestrators.

In [ ]:
def RLGlobalEquityModel(
    trade_tickers: list,
    signal_tickers: list,
    momentum_columns: list,
    trained_agent: PPOAgent = None,
    volatility_penalty: float = 0.5,
) -> Callable[[pl.LazyFrame], pl.LazyFrame]:
    """
    RL-based model that allocates across global equity ETFs based on macro signals.
    
    If trained_agent is None, returns equal-weight allocation (for training baseline).
    If trained_agent is provided, uses the trained policy to generate allocations.
    """
    
    def run_model(lf: pl.LazyFrame) -> pl.LazyFrame:
        # Collect the lazy frame
        df = lf.collect()
        
        # Don't cast date - keep as string to match data
        
        # Get unique dates
        dates = (
            df.select(pl.col("date"))
            .unique()
            .sort("date")
            .to_series()
            .to_list()
        )
        
        # Initialize weights storage
        weights_list = []
        
        for date in dates:
            # Extract state: momentum features from signal tickers
            state_features = []
            date_df = df.filter(pl.col('date') == date)
            
            for signal_ticker in signal_tickers:
                ticker_df = date_df.filter(pl.col('ticker') == signal_ticker)
                
                for momentum_col in momentum_columns:
                    if len(ticker_df) > 0 and momentum_col in ticker_df.columns:
                        val = ticker_df.select(momentum_col).item(0, 0)
                        state_features.append(val if val is not None and not np.isnan(val) else 0.0)
                    else:
                        state_features.append(0.0)
            
            # Convert to numpy array
            state = np.array(state_features, dtype=np.float32)
            
            # Get action from agent or use equal weight
            if trained_agent is not None:
                action_idx, _ = trained_agent.select_action(state, training=False)
                # Create one-hot allocation
                weights = {ticker: 0.0 for ticker in trade_tickers}
                weights[trade_tickers[action_idx]] = 1.0
            else:
                # Equal weight baseline
                weights = {ticker: 1.0 / len(trade_tickers) for ticker in trade_tickers}
            
            # Store weights
            weight_dict = {'date': date}
            weight_dict.update(weights)
            weights_list.append(weight_dict)
        
        # Convert to polars DataFrame
        weights_df = pl.DataFrame(weights_list)
        
        return weights_df.lazy()
    
    return run_model

## 4. Example Usage with trading_model0

Now let's see how to use the RL allocator with the trading_engine orchestrators.

In [ ]:
def train_rl_agent(
    model_state,
    prices,
    trade_tickers: list,
    signal_tickers: list,
    momentum_columns: list,
    num_episodes: int = 100,
    update_frequency: int = 2000,
    volatility_penalty: float = 0.5,
    volatility_window: int = 20,
) -> PPOAgent:
    """
    Train PPO agent on historical data.
    """
    # Initialize agent
    state_dim = len(signal_tickers) * len(momentum_columns)
    action_dim = len(trade_tickers)
    agent = PPOAgent(state_dim, action_dim)
    
    # Clone data
    df = model_state.clone()
    prices_df = prices.clone()
    
    # CRITICAL FIX: Convert model_state dates to strings to match prices dates
    df = df.with_columns(pl.col("date").cast(pl.String))
    
    # Extract dates - they will be strings now
    dates = (
        df.select(pl.col("date"))
        .unique()
        .sort("date")
        .to_series()
        .to_list()
    )
    
    print(f"Training on {len(dates)} days of data...")
    print(f"State dimension: {state_dim}, Action dimension: {action_dim}")
    print(f"Training for {num_episodes} episodes...")
    
    # Pre-compute returns matrix for all trade tickers (vectorized)
    print("Pre-computing returns matrix...")
    returns_dict = {}
    for ticker in trade_tickers:
        if ticker in prices_df.columns:
            ticker_prices = prices_df.select(["date", ticker]).sort("date")
            # Compute returns: (price[t+1] - price[t]) / price[t]
            ticker_returns = ticker_prices.with_columns([
                (pl.col(ticker).shift(-1) / pl.col(ticker) - 1.0).alias("return")
            ])
            # Create date -> return mapping
            returns_dict[ticker] = dict(zip(
                ticker_returns["date"].to_list(),
                ticker_returns["return"].to_list()
            ))
        else:
            print(f"Warning: {ticker} not found in prices_df")
            returns_dict[ticker] = {}
    
    # Verify returns dict has data
    sample_ticker = trade_tickers[0]
    sample_dates = dates[:3]
    sample_returns = [returns_dict[sample_ticker].get(d) for d in sample_dates]
    print(f"Sample returns for {sample_ticker}: {sample_returns}")
    
    # Pre-compute state features for all dates (vectorized)
    print("Pre-computing state features...")
    state_cache = {}
    for date in dates:
        state_features = []
        date_df = df.filter(pl.col('date') == date)
        
        for signal_ticker in signal_tickers:
            ticker_df = date_df.filter(pl.col('ticker') == signal_ticker)
            
            for momentum_col in momentum_columns:
                if len(ticker_df) > 0 and momentum_col in ticker_df.columns:
                    val = ticker_df.select(momentum_col).item(0, 0)
                    state_features.append(val if val is not None and not np.isnan(val) else 0.0)
                else:
                    state_features.append(0.0)
        
        state_cache[date] = np.array(state_features, dtype=np.float32)
    
    print("Starting training loop...")
    
    # Training loop
    episode_rewards = []
    episode_sharpes = []
    
    for episode in range(num_episodes):
        total_reward = 0
        total_return = 0
        timestep = 0
        
        # Track portfolio returns for volatility calculation
        recent_returns = []
        all_returns = []
        
        for i in range(len(dates) - 1):
            date = dates[i]
            
            # Get pre-computed state
            state = state_cache[date]
            
            # Select action
            action_idx, action_prob = agent.select_action(state, training=True)
            selected_ticker = trade_tickers[action_idx]
            
            # Get return from pre-computed returns (much faster)
            daily_return = returns_dict.get(selected_ticker, {}).get(date, 0.0)
            if daily_return is None or np.isnan(daily_return):
                daily_return = 0.0
            
            recent_returns.append(daily_return)
            all_returns.append(daily_return)
            if len(recent_returns) > volatility_window:
                recent_returns.pop(0)
            
            # Compute reward: return - λ * volatility
            if len(recent_returns) >= 5:
                volatility = np.std(recent_returns)
                reward = daily_return - volatility_penalty * volatility
            else:
                reward = daily_return
            
            # Store transition
            is_terminal = (i == len(dates) - 2)
            agent.store_transition(state, action_idx, reward, action_prob, is_terminal)
            
            total_reward += reward
            total_return += daily_return
            timestep += 1
            
            # Update agent periodically
            if timestep % update_frequency == 0 and len(agent.states) > 0:
                agent.update()
        
        # Final update at end of episode
        if len(agent.states) > 0:
            agent.update()
        
        episode_rewards.append(total_return)
        
        # Calculate episode Sharpe ratio
        if len(all_returns) > 1:
            episode_sharpe = np.mean(all_returns) / (np.std(all_returns) + 1e-8) * np.sqrt(252)
            episode_sharpes.append(episode_sharpe)
        else:
            episode_sharpes.append(0.0)
        
        if (episode + 1) % 10 == 0:
            avg_reward = np.mean(episode_rewards[-10:])
            avg_sharpe = np.mean(episode_sharpes[-10:])
            print(f"Episode {episode + 1}/{num_episodes} | "
                  f"Avg Return (last 10): {avg_reward:.4f} | "
                  f"Avg Sharpe (last 10): {avg_sharpe:.4f}")
    
    # Plot training progress
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot episode returns
    ax1.plot(episode_rewards)
    ax1.set_title('Training Progress: Episode Returns', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Total Return')
    ax1.grid(True, alpha=0.3)
    
    # Plot episode Sharpe ratios
    ax2.plot(episode_sharpes)
    ax2.set_title('Training Progress: Episode Sharpe Ratio', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Sharpe Ratio (Annualized)')
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nTraining completed!")
    print(f"Final Sharpe Ratio: {episode_sharpes[-1]:.4f}")
    
    return agent

## 5. Configuration and Data Loading

In [96]:
# Experiment config
universe = [
    "SPY-US", "SLV-US", "GLD-US", "TLT-US", "USO-US", "UNG-US", "IXJ-US",
    "KXI-US", "JXI-US", "IXG-US", "IXN-US", "RXI-US", "MXI-US", "EXI-US",
    "IXC-US", "IEI-US", "SHY-US", "BIL-US", "JPXN-US", "INDA-US", "MCHI-US",
    "EZU-US", "IBIT-US", "ETHA-US", "VIXY-US", "EWJ-US"
]

# Trade tickers (actions)
trade_tickers = ["SPY-US", "EWJ-US", "INDA-US", "MCHI-US", "EZU-US", "BIL-US"]

# Signal tickers (state)
signal_tickers = ["TLT-US", "GLD-US", "USO-US", "SLV-US", "VIXY-US", "IBIT-US"]

# Features
momentum_columns = ["close_momentum_10", "close_momentum_20", "close_momentum_60"]

initial_value = 1_000_000
start_date = dt.date(2004, 1, 2)
end_date = dt.date(2025, 9, 10)

In [97]:
# Load data and create model state
lf = read_data()
model_state, prices = create_model_state(
    lf=lf,
    features=momentum_columns,
    start_date=start_date,
    end_date=end_date,
    universe=universe
)

print("Data loaded successfully!")

Data loaded successfully!


## 6. Train the RL Agent

This will train the PPO agent for the specified number of episodes. Training progress will be displayed every 10 episodes.

In [ ]:
# Train the agent
trained_agent = train_rl_agent(
    model_state=model_state,
    prices=prices,
    trade_tickers=trade_tickers,
    signal_tickers=signal_tickers,
    momentum_columns=momentum_columns,
    num_episodes=50,  # Adjust based on your needs
    update_frequency=2000,
    volatility_penalty=0.5,
)

print("\nTraining completed!")

## 7. Create Model Registry and Run Backtests

Now we'll create a model registry with both the RL model and a baseline equal-weight model for comparison.

In [ ]:
# Create model registry
models_registry = {
    "RL_GlobalEquity": {
        "tickers": trade_tickers + signal_tickers,
        "columns": momentum_columns,
        "function": RLGlobalEquityModel(
            trade_tickers=trade_tickers,
            signal_tickers=signal_tickers,
            momentum_columns=momentum_columns,
            trained_agent=trained_agent,
            volatility_penalty=0.5,
        ),
    },
    "Baseline_EqualWeight": {
        "tickers": trade_tickers + signal_tickers,
        "columns": momentum_columns,
        "function": RLGlobalEquityModel(
            trade_tickers=trade_tickers,
            signal_tickers=signal_tickers,
            momentum_columns=momentum_columns,
            trained_agent=None,  # Equal weight baseline
            volatility_penalty=0.5,
        ),
    },
}

models = ["RL_GlobalEquity", "Baseline_EqualWeight"]

print("Model registry created successfully!")

In [ ]:
# Run model backtests
model_insights = orchestrate_model_backtests(
    model_state=model_state,
    models=models,
    universe=universe,
    registry=models_registry
)

model_simulations = orchestrate_model_simulations(
    prices=prices,
    model_insights=model_insights,
    start_date=start_date,
    end_date=end_date,
    initial_value=initial_value
)

print("\nBacktest Results:")
for model_name in models:
    print(f"\n{model_name}:")
    # Access the canonical backtest results
    metrics = model_simulations[model_name]['backtest_results']['metrics']
    print(metrics)

## 8. Visualize Results

Build equity curves to compare the RL model against the equal-weight baseline.

In [ ]:
# Build equity curves
def build_equity_curve_multi(prices_df, model_simulations, model_names, trade_tickers):
    """
    Build equity curves for multiple models.
    
    Args:
        prices_df: Polars DataFrame in wide format [date, SPY-US, EWJ-US, ...]
        model_simulations: Dict of model simulation results
        model_names: List of model names
        trade_tickers: List of trade ticker names
    """
    # Clone - don't cast date, keep as string
    prices_collected = prices_df.clone()
    
    curves = {}
    
    for model_name in model_names:
        # Access the backtest_results which contains weights and daily returns
        backtest_data = model_simulations[model_name]['backtest_results']
        
        # Get the equity curve directly from backtest results
        if 'equity_curve' in backtest_data:
            equity_values = backtest_data['equity_curve'].to_list()
            dates = backtest_data['dates'].to_list() if 'dates' in backtest_data else list(range(len(equity_values)))
        elif 'daily_value' in backtest_data:
            # Use daily portfolio values
            equity_values = backtest_data['daily_value'].to_list()
            dates = backtest_data['dates'].to_list() if 'dates' in backtest_data else list(range(len(equity_values)))
        else:
            # Fallback: compute from daily returns if available
            if 'daily_return' in backtest_data:
                returns = backtest_data['daily_return'].to_list()
                equity_values = [1.0]
                for ret in returns:
                    equity_values.append(equity_values[-1] * (1 + ret))
                dates = backtest_data['dates'].to_list() if 'dates' in backtest_data else list(range(len(equity_values)))
            else:
                print(f"Warning: No equity data found for {model_name}")
                continue
        
        curves[model_name] = {
            'dates': dates,
            'equity': equity_values
        }
    
    return curves


# Build and plot equity curves
equity_curves = build_equity_curve_multi(prices, model_simulations, models, trade_tickers)

plt.figure(figsize=(14, 7))
for model_name in models:
    if model_name in equity_curves:
        plt.plot(equity_curves[model_name]['dates'], 
                 equity_curves[model_name]['equity'], 
                 label=model_name, linewidth=2)

plt.title("RL Global Equity Model vs Baseline", fontsize=14, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Cumulative Growth of $1")
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Analyze Allocation Patterns

See how the RL agent allocates across different global equity ETFs over time.

In [ ]:
# Analyze how the RL agent allocates across different ETFs
backtest_data = model_simulations['RL_GlobalEquity']['backtest_results']

# Get weights from backtest data
if 'weights' in backtest_data:
    rl_weights = backtest_data['weights']
    if hasattr(rl_weights, 'collect'):
        rl_weights = rl_weights.collect()
elif 'daily_weights' in backtest_data:
    rl_weights = backtest_data['daily_weights']
    if hasattr(rl_weights, 'collect'):
        rl_weights = rl_weights.collect()
else:
    print("Warning: No weights data found in backtest results")
    print("Available keys:", list(backtest_data.keys()))
    rl_weights = None

if rl_weights is not None:
    # Calculate average allocation to each ETF
    print("\nAverage Allocation by RL Agent:")
    for ticker in trade_tickers:
        if ticker in rl_weights.columns:
            avg_weight = rl_weights.select(ticker).mean().item()
            print(f"{ticker:10s}: {avg_weight*100:>6.2f}%")
    
    # Plot allocation over time
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # Get dates if available
    if 'date' in rl_weights.columns:
        dates = rl_weights.select('date')['date'].to_list()
        x_vals = dates
        x_label = "Date"
    else:
        x_vals = range(len(rl_weights))
        x_label = "Time Step"
    
    # Create stacked area chart
    bottom = np.zeros(len(rl_weights))
    for ticker in trade_tickers:
        if ticker in rl_weights.columns:
            weights = rl_weights.select(ticker).to_numpy().flatten()
            ax.fill_between(range(len(weights)), bottom, bottom + weights, label=ticker, alpha=0.7)
            bottom += weights
    
    ax.set_title("RL Agent Allocation Over Time", fontsize=14, fontweight='bold')
    ax.set_xlabel(x_label)
    ax.set_ylabel("Portfolio Weight")
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping allocation plot - no weights data available")

## 10. Save Trained Model

Save the trained agent so you can load it later without retraining.

In [ ]:
# Save the trained agent for future use
torch.save({
    'policy_state_dict': trained_agent.policy.state_dict(),
    'value_state_dict': trained_agent.value.state_dict(),
    'trade_tickers': trade_tickers,
    'signal_tickers': signal_tickers,
    'momentum_columns': momentum_columns,
    'state_dim': len(signal_tickers) * len(momentum_columns),
    'action_dim': len(trade_tickers),
}, 'rl_global_equity_model.pth')

print("\nModel saved to 'rl_global_equity_model.pth'")
print("\nTo load this model later:")
print("checkpoint = torch.load('rl_global_equity_model.pth')")
print("agent = PPOAgent(checkpoint['state_dim'], checkpoint['action_dim'])")
print("agent.policy.load_state_dict(checkpoint['policy_state_dict'])")
print("agent.value.load_state_dict(checkpoint['value_state_dict'])")

In [ ]:
# DEBUG: Check why returns are 0
# Run this cell to diagnose the returns computation issue

print("="*80)
print("DEBUGGING RETURNS")
print("="*80)

# Check prices structure
print(f"\nPrices DataFrame:")
print(f"  Shape: {prices.shape}")
print(f"  Columns: {list(prices.columns)[:15]}...")
print(f"  Date type: {prices.schema['date']}")

# Get sample dates
dates = prices.select(pl.col("date")).unique().sort("date").to_series().to_list()
print(f"\nDates: {len(dates)} total")
print(f"  First: {dates[0]}")
print(f"  Last: {dates[-1]}")

# Check one ticker in detail
test_ticker = "SPY-US"
print(f"\nTesting {test_ticker}:")

if test_ticker in prices.columns:
    ticker_df = prices.select(["date", test_ticker]).sort("date")
    print(f"  Found! Rows: {len(ticker_df)}")
    print(f"  Sample data:")
    print(ticker_df.head(5))
    
    # Compute returns
    with_returns = ticker_df.with_columns([
        (pl.col(test_ticker).shift(-1) / pl.col(test_ticker) - 1.0).alias("return")
    ])
    print(f"\n  With returns:")
    print(with_returns.head(5))
    
    # Check for valid returns
    valid_returns = with_returns.filter(
        pl.col("return").is_not_null() & pl.col("return").is_not_nan()
    )
    print(f"\n  Valid returns: {len(valid_returns)} / {len(with_returns)}")
    
    if len(valid_returns) > 0:
        print(f"  Mean return: {valid_returns.select('return').mean().item():.6f}")
        print(f"  Std return: {valid_returns.select('return').std().item():.6f}")
else:
    print(f"  ❌ NOT FOUND in prices columns!")

print(f"\n{'='*80}")

In [ ]:
# DEBUG: Test the returns dictionary lookup
print("="*80)
print("TESTING RETURNS DICTIONARY LOOKUP")
print("="*80)

# Simulate what the training loop does
prices_df = prices.clone()

# Get dates from model_state (what training uses)
model_dates = (
    model_state.select(pl.col("date"))
    .unique()
    .sort("date")
    .to_series()
    .to_list()
)

# Get dates from prices (what we use to build returns_dict)
price_dates = (
    prices_df.select(pl.col("date"))
    .unique()
    .sort("date")
    .to_series()
    .to_list()
)

print(f"\nModel dates: {len(model_dates)} total")
print(f"  First 3: {model_dates[:3]}")
print(f"  Type: {type(model_dates[0])}")

print(f"\nPrice dates: {len(price_dates)} total")
print(f"  First 3: {price_dates[:3]}")
print(f"  Type: {type(price_dates[0])}")

# Check if they match
if model_dates[:5] == price_dates[:5]:
    print("\n✓ Dates match!")
else:
    print("\n❌ Dates DON'T match!")
    print(f"  Model date: '{model_dates[0]}' (type={type(model_dates[0])})")
    print(f"  Price date: '{price_dates[0]}' (type={type(price_dates[0])})")

# Build returns dict like training does
test_ticker = "SPY-US"
ticker_prices = prices_df.select(["date", test_ticker]).sort("date")
ticker_returns = ticker_prices.with_columns([
    (pl.col(test_ticker).shift(-1) / pl.col(test_ticker) - 1.0).alias("return")
])

returns_dict = dict(zip(
    ticker_returns["date"].to_list(),
    ticker_returns["return"].to_list()
))

print(f"\nReturns dict created: {len(returns_dict)} entries")

# Test lookups using model_state dates (what training actually uses)
print(f"\nTesting lookups with model_state dates:")
for i, date in enumerate(model_dates[:5]):
    ret = returns_dict.get(date)
    print(f"  Date {i}: '{date}' -> {ret}")

print(f"\n{'='*80}")

In [ ]:
# Compare RL Model vs Baselines
print("="*80)
print("BACKTEST COMPARISON: RL MODEL vs BASELINES")
print("="*80)

# First, inspect what's in the backtest results
print("\nInspecting backtest results structure...")
sample_model = models[0]
backtest_data = model_simulations[sample_model]['backtest_results']
print(f"Type: {type(backtest_data)}")

if isinstance(backtest_data, pl.DataFrame):
    print(f"Columns: {backtest_data.columns}")
    print(f"\nSample data:")
    print(backtest_data.head())
else:
    print(f"Keys: {list(backtest_data.keys()) if hasattr(backtest_data, 'keys') else 'N/A'}")

# Show metrics for each model
print("\n" + "="*80)
print("METRICS COMPARISON")
print("="*80)

for model_name in models:
    print(f"\n{model_name}:")
    print("-" * 40)
    
    backtest_data = model_simulations[model_name]['backtest_results']
    
    # If it's a DataFrame, look for a metrics row or compute metrics
    if isinstance(backtest_data, pl.DataFrame):
        # Check if there's a 'metric' and 'value' column (common format)
        if 'metric' in backtest_data.columns and 'value' in backtest_data.columns:
            for row in backtest_data.iter_rows(named=True):
                print(f"  {row['metric']}: {row['value']:.6f}")
        else:
            # Calculate basic metrics from the data
            if 'daily_return' in backtest_data.columns:
                returns = backtest_data['daily_return'].to_numpy()
                returns = returns[~np.isnan(returns)]
                
                if len(returns) > 0:
                    total_return = (1 + returns).prod() - 1
                    ann_return = ((1 + total_return) ** (252 / len(returns))) - 1
                    ann_vol = returns.std() * np.sqrt(252)
                    sharpe = (ann_return / ann_vol) if ann_vol > 0 else 0
                    
                    print(f"  Total Return: {total_return:.6f}")
                    print(f"  Annualized Return: {ann_return:.6f}")
                    print(f"  Annualized Volatility: {ann_vol:.6f}")
                    print(f"  Sharpe Ratio: {sharpe:.6f}")

# Extract and plot equity curves
print("\n" + "="*80)
print("EQUITY CURVES")
print("="*80)

fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, model_name in enumerate(models):
    backtest_data = model_simulations[model_name]['backtest_results']
    
    if isinstance(backtest_data, pl.DataFrame):
        # Look for portfolio value or daily value column
        value_col = None
        if 'portfolio_value' in backtest_data.columns:
            value_col = 'portfolio_value'
        elif 'daily_value' in backtest_data.columns:
            value_col = 'daily_value'
        elif 'value' in backtest_data.columns:
            value_col = 'value'
        
        if value_col:
            portfolio_values = backtest_data[value_col].to_list()
            
            # Normalize to start at $1
            if len(portfolio_values) > 0 and portfolio_values[0] > 0:
                normalized_values = [v / portfolio_values[0] for v in portfolio_values]
                
                # Get dates if available
                if 'date' in backtest_data.columns:
                    dates = backtest_data['date'].to_list()
                else:
                    dates = range(len(normalized_values))
                
                # Plot
                label = model_name.replace("_", " ")
                ax.plot(dates, normalized_values, label=label, linewidth=2, color=colors[i % len(colors)])
                
                # Print final performance
                final_value = normalized_values[-1]
                total_return = (final_value - 1.0) * 100
                print(f"\n{model_name}:")
                print(f"  Final Value: ${final_value:.2f}")
                print(f"  Total Return: {total_return:.2f}%")
        else:
            # Try to build from daily returns
            if 'daily_return' in backtest_data.columns:
                returns = backtest_data['daily_return'].to_list()
                
                # Build equity curve
                equity = [1.0]
                for ret in returns:
                    if ret is not None and not np.isnan(ret):
                        equity.append(equity[-1] * (1 + ret))
                    else:
                        equity.append(equity[-1])
                
                # Get dates
                if 'date' in backtest_data.columns:
                    dates = backtest_data['date'].to_list()[:len(equity)]
                else:
                    dates = range(len(equity))
                
                # Plot
                label = model_name.replace("_", " ")
                ax.plot(dates, equity, label=label, linewidth=2, color=colors[i % len(colors)])
                
                # Print final performance
                final_value = equity[-1]
                total_return = (final_value - 1.0) * 100
                print(f"\n{model_name}:")
                print(f"  Final Value: ${final_value:.2f}")
                print(f"  Total Return: {total_return:.2f}%")
            else:
                print(f"\nWARNING: Could not plot {model_name}")
                print(f"Available columns: {backtest_data.columns[:10]}")

ax.set_title("RL Global Equity Model vs Equal-Weight Baseline", fontsize=14, fontweight='bold')
ax.set_xlabel("Date")
ax.set_ylabel("Portfolio Value (Starting with $1)")
ax.set_yscale('log')
ax.legend(loc='best', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ Comparison complete!")